# Machine Learning Development Process

## Introduction

Building a machine learning model rarely ends after the first training run. In practice, models often fail to achieve the desired level of performance, requiring practitioners to investigate errors, identify weaknesses, and decide which improvements are most likely to be effective.

Successful machine learning development is therefore an iterative process. Rather than making random changes, practitioners systematically evaluate model performance, perform error analysis, and use evidence to guide future improvements.

In this notebook, we explore the machine learning development process through a series of experiments focused on model evaluation, error analysis, and data-centric improvement strategies.

## Objective

The goal of this notebook is to understand how machine learning practitioners systematically improve model performance.

Specifically, we aim to:

- Understand the iterative nature of machine learning development.
- Learn how error analysis can reveal model weaknesses.
- Investigate how data-centric improvements influence performance.
- Explore when transfer learning may be beneficial.
- Discuss fairness and ethical considerations during model development.
- Develop a structured framework for deciding what to try next when a model performs poorly.

### Guiding Question

How can machine learning practitioners systematically diagnose and improve a poorly performing model?

## Background Theory

Developing machine learning systems involves far more than selecting an algorithm and training a model. Performance improvements typically emerge through an iterative process of evaluation, diagnosis, and refinement.

When a model performs poorly, practitioners must determine the underlying cause before deciding on corrective actions. Potential improvements may include collecting additional data, engineering new features, reducing overfitting, applying transfer learning, or addressing issues within the dataset itself.

The machine learning development process provides a structured framework for making these decisions based on evidence rather than intuition.

### Iterative Machine Learning Development

Machine learning development is often an iterative cycle:

1. Train a baseline model.
2. Evaluate performance.
3. Perform error analysis.
4. Identify the most significant sources of error.
5. Implement targeted improvements.
6. Re-evaluate the updated system.

This process continues until performance reaches an acceptable level or additional improvements become impractical.

### Error Analysis

Error analysis involves examining incorrect predictions to identify systematic patterns in model failures.

Rather than focusing solely on aggregate metrics such as accuracy, practitioners inspect specific mistakes and attempt to answer questions such as:

- Which examples are most difficult for the model?
- Are certain classes more frequently misclassified?
- Is the dataset missing important examples?
- Are there labeling issues or data quality problems?

The insights gained through error analysis often guide the next stage of model improvement.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [2]:
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_classes=2,
    random_state=42
)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size= 0.2,
    random_state= 42
)
print("Training samples: ", X_train.shape)
print("Test samples: ", X_test.shape)

Training samples:  (800, 20)
Test samples:  (200, 20)


## Baseline Model

The first step in the machine learning development process is to establish a baseline model.

A baseline provides a reference point against which future improvements can be measured. Rather than immediately searching for complex solutions, practitioners first evaluate how well a simple model performs and identify its weaknesses through error analysis.

In this experiment, Logistic Regression is used as the baseline classifier.

In [4]:
baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test)

## Baseline Evaluation

To assess the baseline model, multiple evaluation metrics are considered.

Accuracy provides an overall measure of performance, while precision, recall, and F1-score offer additional insight into classification quality. A confusion matrix is also examined to understand the types of mistakes made by the model.

In [5]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

Accuracy:  0.7950
Precision: 0.7767
Recall:    0.8163
F1 Score:  0.7960


In [6]:
cm = confusion_matrix(
    y_test,
    y_pred
)

cm

array([[79, 23],
       [18, 80]])

## Error Analysis

Aggregate metrics provide useful information, but they do not explain why a model makes mistakes.

To better understand model behavior, individual predictions can be examined. By analyzing incorrect predictions, practitioners can identify systematic weaknesses and determine which improvements are most likely to increase performance.

The following analysis focuses on misclassified examples produced by the baseline model.

In [7]:
error_df = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

error_df["Correct"] = (
    error_df["Actual"]
    ==
    error_df["Predicted"]
)

error_df.head()

,Actual,Predicted,Correct
0,0,1,False
1,0,0,True
2,1,0,False
3,1,1,True
4,0,0,True


In [8]:
errors = error_df[
    error_df["Correct"] == False
]

print(
    f"Total Misclassified Examples: {len(errors)}"
)

errors.head()

Total Misclassified Examples: 41


,Actual,Predicted,Correct
0,0,1,False
2,1,0,False
6,0,1,False
10,1,0,False
15,1,0,False


### Error Analysis Findings

The baseline model achieved reasonable performance but still produced a substantial number of classification errors.

The confusion matrix reveals both false positives and false negatives. Specifically, the model incorrectly classified 23 negative examples as positive and 18 positive examples as negative.

Rather than immediately selecting a more complex model, a practitioner should investigate these mistakes and consider whether additional data, improved features, or alternative modeling approaches could reduce the observed errors.

This process forms the basis of data-centric machine learning development.

## Deciding What to Try Next

After evaluating the baseline model, a machine learning practitioner must decide which improvement is most likely to increase performance.

Several possible strategies include:

- Collecting additional training data.
- Engineering more informative features.
- Addressing data quality issues.
- Applying regularization.
- Using transfer learning.
- Modifying the model architecture.

The appropriate choice depends on the results of evaluation and error analysis rather than intuition alone.

## Error Analysis Case Study

Suppose inspection of misclassified examples reveals that many errors occur in examples that are difficult to distinguish using the available features.

In this scenario, simply training the same model for longer is unlikely to produce substantial improvements. Instead, a practitioner may consider collecting additional data, designing better features, or obtaining higher-quality labels.

This illustrates the importance of understanding model failures before selecting an improvement strategy.

## Data-Centric Machine Learning

Traditional machine learning workflows often focus on improving algorithms. However, many practical gains come from improving the data itself.

Examples of data-centric improvements include:

- Collecting additional training examples.
- Correcting mislabeled examples.
- Improving data quality.
- Increasing representation of underrepresented cases.
- Reducing noise in the dataset.

In many real-world projects, improving the dataset can produce larger gains than changing the model.

### Bias and Variance Diagnosis

Error analysis tells us *that* a model is making mistakes. Bias and variance diagnosis tells us *why*, which determines what fix is actually worth trying.

- **High bias (underfitting):** training error itself is high, and the cv error is close to the training error. The model is too simple to capture the underlying pattern. Collecting more data will not help much; better features or a more expressive model usually will.
- **High variance (overfitting):** training error is low, but cv error is much higher. The model has memorized noise in the training set. More data, regularization, or simplifying the model tends to help.

To diagnose this properly, the dataset needs three splits: a training set, a cross-validation (CV) set used to tune decisions, and a test set held out until the very end. The original 80/20 split used for the baseline model only had train and test, so a CV set is introduced here for diagnostic purposes.

In [9]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train, X_cv, y_train, y_cv = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,   # 0.25 × 0.8 = 0.2
    random_state=42
)

print("Train:", X_train.shape)
print("CV:", X_cv.shape)
print("Test:", X_test.shape)

Train: (600, 20)
CV: (200, 20)
Test: (200, 20)


In [10]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

train_acc = accuracy_score(
    y_train,
    model.predict(X_train)
)

cv_acc = accuracy_score(
    y_cv,
    model.predict(X_cv)
)

print("Training Accuracy:", train_acc)
print("CV Accuracy:", cv_acc)

Training Accuracy: 0.8716666666666667
CV Accuracy: 0.845


# Bias-Variance Diagnosis

We trained a Logistic Regression model and evaluated it on both the training and cross-validation sets.

Results:

- Training Accuracy = 87.17%
- Cross-Validation Accuracy = 84.50%

Difference:

87.17% - 84.50% = 2.67%

## Interpretation

The training and cross-validation accuracies are relatively close, indicating that the model generalizes reasonably well to unseen data.

There is no strong evidence of high variance because the performance gap is small.

Likewise, there is no obvious indication of severe high bias because both accuracies are reasonably high.

Therefore, the model appears to have achieved a good balance between bias and variance.

In [11]:
train_acc = 0.8716666666666667
cv_acc = 0.845

difference = train_acc - cv_acc

print(f"Training Accuracy: {train_acc:.4f}")
print(f"CV Accuracy: {cv_acc:.4f}")
print(f"Difference: {difference:.4f}")

if difference > 0.10:
    print("\nDiagnosis: High Variance")
elif train_acc < 0.80 and cv_acc < 0.80:
    print("\nDiagnosis: High Bias")
else:
    print("\nDiagnosis: Reasonable Fit")

Training Accuracy: 0.8717
CV Accuracy: 0.8450
Difference: 0.0267

Diagnosis: Reasonable Fit


# What Should We Try Next?

The machine learning development process focuses on making decisions based on evidence rather than guessing.

If we observed:

### High Bias

Possible actions:

- Add more informative features
- Use a more complex model
- Reduce regularization

### High Variance

Possible actions:

- Collect more training data
- Increase regularization
- Simplify the model

### Current Situation

Our model does not show strong signs of either high bias or high variance.

At this stage, additional improvements would require experimentation and further analysis of the data.

# Key Takeaways

- Accuracy alone is not enough to evaluate a model.
- Error analysis helps identify where a model fails.
- Training, cross-validation, and test sets serve different purposes.
- Bias and variance require different solutions.
- Machine learning development should be driven by evidence and diagnostics rather than random trial and error.

This notebook demonstrated the basic workflow used when developing machine learning systems:
1. Train a model
2. Evaluate performance
3. Analyze errors
4. Diagnose bias and variance
5. Decide what to try next